# Midi Audio Synthesis

This notebook is used to synthesize the .mid files from Lakh MIDI dataset to be used for running the ablation studies on the out-of-domain evaluation of LoRA fine-tuned SAM-Audio model on Turkish musical instrument (the 00.1 notebook). The .mid file were available at https://www.kaggle.com/datasets/imsparsh/lakh-midi-clean. We will use the `midi2audio` library to synthesize the .mid files into .wav files. The `midi2audio` library is a Python wrapper for the FluidSynth software synthesizer, which can render MIDI files using SoundFont files.

The soundfont files were available to be downloaded from community site (https://www.keyfimuzik.net/fl-studio/106943-turkish-instruments-soundfont-sf2-dev-arsiv.html) (big thanks to [Dmex35](https://www.keyfimuzik.net/members/363067-dmex35.html)) and we have added the metadata of the soundfont files in `data/soundfonts_programs.json`. We will use these soundfont files to synthesize the MIDI files into audio.

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download(
    "imsparsh/lakh-midi-clean", output_dir="../data/midis/lmd"
)

print("Path to dataset files:", path)

Path to dataset files: ../data/midis/lmd


In [3]:
from midi2audio import FluidSynth
import os
from contextlib import contextmanager, redirect_stderr, redirect_stdout


@contextmanager
def suppress_stdout_stderr():
    """A context manager that redirects stdout and stderr to devnull"""
    with open(os.devnull, "w") as fnull:
        with redirect_stderr(fnull), redirect_stdout(fnull):
            yield


def synthesize_lmd_midi(midi_path, soundfont_path, output_wav_path):
    """
    Renders a microtonal SymbTr MIDI file into a .wav file using a specific SoundFont.
    """
    # Initialize FluidSynth with your downloaded Turkish .sf2 SoundFont
    fs = FluidSynth(sound_font=soundfont_path)

    # Check if files exist to prevent errors
    if not os.path.exists(midi_path):
        print(f"Error: MIDI file not found at {midi_path}")
        return
    if not os.path.exists(soundfont_path):
        print(f"Error: SoundFont not found at {soundfont_path}")
        return

    # Render the audio stem
    # print(f"Synthesizing {midi_path}...")

    # Wrap the call to hide warnings/logs
    with suppress_stdout_stderr():
        fs.midi_to_audio(midi_path, output_wav_path)

    # print(f"Success! Audio saved to {output_wav_path}")


In [4]:
import mido


def inspect_midi_data(midi_path):
    """
    Loads a MIDI file and prints its internal messages,
    allowing you to see the baked-in microtonal pitch bends.
    """
    # Load the MIDI file
    mid = mido.MidiFile(midi_path)

    print(f"Inspecting: {midi_path}")
    print(f"Total Playback Time: {mid.length:.2f} seconds")

    # Iterate through the tracks and messages
    for i, track in enumerate(mid.tracks):
        print(f"\n--- Track {i}: {track.name} ---")

        # Print the first 15 messages just to see the structure
        for msg in track[:15]:
            # This is where you will see 'note_on', 'note_off', and crucial 'pitchwheel' events
            print(msg)

In [ ]:
inspect_midi_data(
    "/teamspace/studios/this_studio/anatolian-SAM/data/midis/lmd/ABBA/The_Winner_Takes_It_All.1.mid"
)

Inspecting: /teamspace/studios/this_studio/anatolian-SAM/data/midis/lmd/ABBA/The_Winner_Takes_It_All.1.mid
Total Playback Time: 253.12 seconds

--- Track 0: The Winne ---
MetaMessage('track_name', name='The Winne', time=0)
MetaMessage('set_tempo', tempo=468750, time=0)
MetaMessage('time_signature', numerator=4, denominator=4, clocks_per_click=24, notated_32nd_notes_per_beat=8, time=0)
sysex data=(65,16,66,18,64,0,127,0,65) time=1536
sysex data=(65,16,66,18,64,1,16,4,0,3,3,3,3,3,3,0,0,0,0,0,0,0,0,25) time=96
sysex data=(65,16,66,18,64,1,48,4,11) time=36
sysex data=(65,16,66,18,64,1,56,2,5) time=2
control_change channel=0 control=121 value=127 time=58
control_change channel=1 control=121 value=127 time=2
control_change channel=2 control=121 value=127 time=2
control_change channel=3 control=121 value=127 time=2
control_change channel=4 control=121 value=127 time=2
control_change channel=5 control=121 value=127 time=2
control_change channel=6 control=121 value=127 time=2
control_change cha

## Batch Processing

In [6]:
DATA_DIR = "../data"

In [7]:
# read json file to see the soundfont programs
import json

with open(os.path.join(DATA_DIR, "soundfonts_programs_ww.json"), "r") as f:
    soundfont_programs = json.load(f)

soundfont_programs

{'soundfonts': [{'name': 'steel_string_guitar',
   'filename': 'FSS-SteelStringGuitar-small-20200521.sf2',
   'description': 'Steel-String Acoustic Guitar soundfont, the Plucked Lute (Metal Strings) class',
   'program': '000'},
  {'name': 'harp',
   'filename': 'ConcertHarp-small-20200702.sf2',
   'description': 'Orchestral Harp soundfont, the Plucked Zither class',
   'program': '000'},
  {'name': 'recorder',
   'filename': 'Recorder-20201205.sf2',
   'description': 'Recorder soundfont, the Woodwind (End-blown Flute) class',
   'program': '000'},
  {'name': 'synth_strings',
   'filename': 'SynthStrings1 20200528.sf2',
   'description': 'Synth Strings 1 soundfont, the Bowed Strings envelope class',
   'program': '000'},
  {'name': 'nylon_guitar',
   'filename': 'SpanishClassicalGuitar-20190618.sf2',
   'description': 'Nylon-String Acoustic Guitar soundfont, The Plucked Lute (Nylon/Gut Strings) class',
   'program': '000'},
  {'name': 'bagpipes',
   'filename': 'Bagpipe-small-20221204.

In [ ]:
from tqdm.autonotebook import tqdm
from glob import glob
import random


LIMITS = 3  # set to None to synthesize all the midi files

lmd_midis = glob(os.path.join(DATA_DIR, "midis", "lmd", "**", "*.mid"), recursive=True)
midi_samples = random.sample(lmd_midis, min(LIMITS, len(lmd_midis)))
midi_samples.append("../data/midis/lmd/ABBA/The_Winner_Takes_It_All.1.mid")
midi_samples.append(
    "../data/midis/lmd/Tears_for_Fears/Everybody_Wants_To_Rule_The_World.2.mid"
)

for sf in soundfont_programs["soundfonts"]:
    for midi in tqdm(
        midi_samples,
        desc=f"Synthesizing {sf['name']}",
    ):
        # synthesize the fixed midi file with the corresponding soundfont
        if not os.path.exists(os.path.join(DATA_DIR, "synthesized", sf["name"])):
            os.makedirs(
                os.path.join(DATA_DIR, "synthesized", sf["name"]), exist_ok=True
            )

        # get the midi filename without extension
        midi_filename = os.path.split(midi)[-1][:-4]
        output_wav_path = os.path.join(
            DATA_DIR, "synthesized", sf["name"], f"{midi_filename}.wav"
        )

        synthesize_lmd_midi(
            midi,
            os.path.join(DATA_DIR, "soundfonts", sf["filename"]),
            output_wav_path,
        )

Synthesizing steel_string_guitar:   0%|          | 0/5 [00:00<?, ?it/s]fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
fluidsynth: warning: Instrument not found on channel 0 [bank=0 prog=67], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 3 [bank=0 prog=28], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 1 [bank=0 prog=33], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 5 [bank=0 prog=67], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 7 [bank=0 prog=67], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 6 [bank=0 prog=17], substituted [bank=0 prog=0]


fluidsynth: warning: Instrument not found on channel 2 [bank=0 prog=67], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 4 [bank=0 prog=66], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 0 [bank=0 prog=64], substituted [bank=0 prog=0]
Synthesizing steel_string_guitar:  20%|██        | 1/5 [00:06<00:25,  6.29s/it]

FluidSynth runtime version 2.3.4
Copyright (C) 2000-2023 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '../data/synthesized/steel_string_guitar/Locomotion.wav'..


fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
fluidsynth: warning: Instrument not found on channel 1 [bank=0 prog=38], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 2 [bank=0 prog=81], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 3 [bank=0 prog=80], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 4 [bank=0 prog=74], substituted [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
Synthesizing steel_string_guitar:  40%|████      | 2/5 [00:12<00:18,  6.22s/it]

FluidSynth runtime version 2.3.4
Copyright (C) 2000-2023 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '../data/synthesized/steel_string_guitar/Rode_schoentjes.wav'..


...

fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
fluidsynth: warning: Instrument not found on channel 1 [bank=0 prog=38], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 2 [bank=0 prog=81], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 3 [bank=0 prog=80], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 4 [bank=0 prog=74], substituted [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
Synthesizing harp:  40%|████      | 2/5 [00:23<00:34, 11.65s/it]

FluidSynth runtime version 2.3.4
Copyright (C) 2000-2023 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '../data/synthesized/harp/Rode_schoentjes.wav'..


fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
fluidsynth: warning: Instrument not found on channel 5 [bank=0 prog=66], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 1 [bank=0 prog=35], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 6 [bank=0 prog=27], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 7 [bank=0 prog=52], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 2 [bank=0 prog=27], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 3 [bank=0 prog=61], substituted [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
fluidsynth: warning: Instrument not found on channel 4 [bank=0 prog=65], substituted [bank=0 prog=0]
Synthesizing harp:  60%|██████    | 3/5 [00:34<00:22, 11.37s/it]

FluidSynth runtime version 2.3.4
Copyright (C) 2000-2023 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '../data/synthesized/harp/See_You_Later_Alligator.wav'..


fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
fluidsynth: warning: Instrument not found on channel 1 [bank=0 prog=35], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 2 [bank=0 prog=6], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 3 [bank=0 prog=1], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 4 [bank=0 prog=42], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 5 [bank=0 prog=52], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 6 [bank=0 prog=48], substituted [bank=0 prog=0]
fluidsynth: warning: Instrument not found on channel 7 [bank=0 prog=50], substituted [bank=0 prog=0]
fluidsynth: warning: No preset found on channel 9 [bank=128 prog=0]
